# Step 11 — Download MJO RMM Labels
**Project:** ENSO-BSISO Self-Supervised Learning — MJO Extension  
**Author:** Jiayi (jh9141@nyu.edu)

Downloads and parses the Wheeler & Hendon (2004) Real-time Multivariate MJO (RMM) index, then attaches a monthly ENSO category to each day.

**Design decisions (Session 20, 2026-05-12):**
- Season: **all-year** (1979–2023, ~16,425 days)
- ENSO definition: **monthly Niño 3.4** > +0.5 K → El Niño; < −0.5 K → La Niña; else Neutral  
  (same threshold used in BSISO project but applied per month, not JJA mean)
- Weak-MJO flag: rows with `amplitude < 1.0` are kept but flagged (used for filtering in nb14/nb15)

**Output:** `BSISO_SSL_Project/MJO/data/raw/rmm_labels.csv`  
Columns: `date, rmm1, rmm2, phase, amplitude, enso_category, weak_mjo`

---

## Cell 1 — Mount Google Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import requests
import pandas as pd
import numpy as np

PROJECT_DIR = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_DIR     = f'{PROJECT_DIR}/MJO'
RAW_DIR     = f'{MJO_DIR}/data/raw'
BSISO_RAW   = f'{PROJECT_DIR}/data/raw'   # reuse noaa_enso file if already downloaded

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(f'{MJO_DIR}/data/processed', exist_ok=True)
os.makedirs(f'{MJO_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{MJO_DIR}/results/sup', exist_ok=True)
os.makedirs(f'{MJO_DIR}/results/ssl', exist_ok=True)

print('Google Drive mounted.')
print(f'MJO raw folder: {RAW_DIR}')

## Cell 2 — Download RMM Index from BoM

Source: Bureau of Meteorology (Australia)  
Format (space-separated):  
`year  month  day  RMM1  RMM2  phase  amplitude  [flag]`  
- `phase = 9` means missing / weak MJO (amplitude < 1)  
- `flag = baddata` marks unreliable rows

In [ ]:
RMM_URL  = 'http://www.bom.gov.au/climate/mjo/graphics/rmm.74toRealtime.txt'
rmm_path = f'{RAW_DIR}/rmm_raw.txt'

if os.path.exists(rmm_path):
    print(f'Already downloaded: {rmm_path}')
else:
    print(f'Downloading RMM index from BoM...')
    r = requests.get(RMM_URL, timeout=60)
    r.raise_for_status()
    with open(rmm_path, 'w') as f:
        f.write(r.text)
    print(f'Saved: {rmm_path}  ({os.path.getsize(rmm_path)/1e3:.0f} KB)')

# Preview
print('\nFirst 8 lines:')
with open(rmm_path) as f:
    for i, line in enumerate(f):
        print(f'  {line.rstrip()}')
        if i >= 7:
            break

## Cell 3 — Parse RMM File into DataFrame

In [ ]:
rows = []
with open(rmm_path) as f:
    for line in f:
        line = line.strip()
        # skip header lines (start with letters)
        if not line or line[0].isalpha():
            continue
        parts = line.split()
        if len(parts) < 7:
            continue
        try:
            year, month, day = int(parts[0]), int(parts[1]), int(parts[2])
            rmm1, rmm2       = float(parts[3]), float(parts[4])
            phase             = int(parts[5])
            amplitude         = float(parts[6])
            flag              = parts[7] if len(parts) > 7 else 'good'
        except ValueError:
            continue
        rows.append(dict(year=year, month=month, day=day,
                         rmm1=rmm1, rmm2=rmm2, phase=phase,
                         amplitude=amplitude, flag=flag))

df_rmm = pd.DataFrame(rows)
df_rmm['date'] = pd.to_datetime(df_rmm[['year', 'month', 'day']])

print(f'Total RMM records parsed: {len(df_rmm)}')
print(f'Date range: {df_rmm["date"].min().date()} to {df_rmm["date"].max().date()}')
print(f'Baddata rows: {(df_rmm["flag"] == "baddata").sum()}')
print(f'Phase 9 rows (weak/missing MJO): {(df_rmm["phase"] == 9).sum()}')
print()
print(df_rmm[['date', 'rmm1', 'rmm2', 'phase', 'amplitude', 'flag']].head(8))

## Cell 4 — Filter to 1979–2023 (All Year)

In [ ]:
# Keep 1979-2023, drop baddata rows
df = df_rmm[
    (df_rmm['date'].dt.year >= 1979) &
    (df_rmm['date'].dt.year <= 2023) &
    (df_rmm['flag'] != 'baddata')
].copy().reset_index(drop=True)

# Add weak_mjo flag (amplitude < 1.0 OR phase == 9)
df['weak_mjo'] = (df['amplitude'] < 1.0) | (df['phase'] == 9)

print(f'After filter: {len(df)} days  (expected ~16,425 for 45 years)')
print(f'  Active MJO days (amplitude >= 1.0): {(~df["weak_mjo"]).sum()}')
print(f'  Weak/missing MJO days:              {df["weak_mjo"].sum()}')
print()

# Phase distribution (active MJO only)
active = df[~df['weak_mjo']]
print(f'Phase distribution — active MJO days ({len(active)} total):')
print(active['phase'].value_counts().sort_index())

## Cell 5 — Download NOAA Niño 3.4 Monthly Index

For all-year data we use **monthly** Niño 3.4 (not JJA mean).  
Each calendar day is assigned the Niño 3.4 anomaly of its calendar month.

In [ ]:
NINO_URL  = 'https://www.cpc.ncep.noaa.gov/data/indices/ersst5.nino.mth.91-20.ascii'
nino_path = f'{RAW_DIR}/nino34_monthly.txt'

# Reuse existing file from BSISO project if available
bsiso_nino = f'{BSISO_RAW}/nino34_monthly.txt'
if os.path.exists(bsiso_nino) and not os.path.exists(nino_path):
    import shutil
    shutil.copy(bsiso_nino, nino_path)
    print(f'Copied existing Nino3.4 file from BSISO project.')
elif os.path.exists(nino_path):
    print(f'File already exists: {nino_path}')
else:
    print(f'Downloading Nino3.4 from NOAA CPC...')
    r = requests.get(NINO_URL, timeout=30)
    r.raise_for_status()
    with open(nino_path, 'w') as f:
        f.write(r.text)
    print(f'Saved: {nino_path}')

# Preview
print('\nFirst 5 lines:')
with open(nino_path) as f:
    for i, line in enumerate(f):
        print(f'  {line.rstrip()}')
        if i >= 4:
            break

## Cell 6 — Parse Niño 3.4 Monthly Index

In [ ]:
# Format: YR  MON  NINO1+2  ANOM  NINO3  ANOM  NINO4  ANOM  NINO3.4  ANOM
# We need columns 0 (year), 1 (month), 9 (Nino3.4 anomaly)

nino_rows = []
with open(nino_path) as f:
    for line in f:
        line = line.strip()
        if not line or line[0].isalpha():
            continue
        parts = line.split()
        if len(parts) >= 10:
            try:
                nino_rows.append({
                    'year':        int(parts[0]),
                    'month':       int(parts[1]),
                    'nino34_anom': float(parts[9])
                })
            except ValueError:
                continue

df_nino = pd.DataFrame(nino_rows)
print(f'Nino3.4 monthly records: {len(df_nino)}')
print(f'Year range: {df_nino["year"].min()} – {df_nino["year"].max()}')

def classify_enso(v):
    if v >= 0.5:  return 'El Nino'
    if v <= -0.5: return 'La Nina'
    return 'Neutral'

df_nino['enso_category'] = df_nino['nino34_anom'].apply(classify_enso)

# Filter to 1979-2023 and show distribution
df_nino_filt = df_nino[(df_nino['year'] >= 1979) & (df_nino['year'] <= 2023)]
print(f'\nMonthly ENSO distribution (1979-2023, {len(df_nino_filt)} months):')
print(df_nino_filt['enso_category'].value_counts())

## Cell 7 — Merge RMM + ENSO → rmm_labels.csv

Each day is assigned the Niño 3.4 anomaly of its calendar month.

In [ ]:
# Add year/month keys to RMM dataframe for merging
df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month

# Merge
df_labels = df.merge(
    df_nino[['year', 'month', 'nino34_anom', 'enso_category']],
    on=['year', 'month'],
    how='left'
)

n_missing = df_labels['enso_category'].isna().sum()
if n_missing > 0:
    print(f'WARNING: {n_missing} rows missing ENSO category — check year/month coverage')
    print(df_labels[df_labels['enso_category'].isna()][['date']].head(10))
else:
    print('All rows have ENSO category assigned.')

# Final columns
df_labels = df_labels[[
    'date', 'rmm1', 'rmm2', 'phase', 'amplitude', 'enso_category', 'nino34_anom', 'weak_mjo'
]]

out_path = f'{RAW_DIR}/rmm_labels.csv'
df_labels.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Shape: {df_labels.shape}')
print()
print(df_labels.head(8))

## Cell 8 — Verification: Summary Statistics

In [ ]:
df = pd.read_csv(out_path, parse_dates=['date'])
active = df[~df['weak_mjo']]

print('=' * 60)
print('RMM LABELS VERIFICATION REPORT')
print('=' * 60)
print(f'Total days (1979-2023):    {len(df)}')
print(f'Date range:                {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Active MJO (ampl >= 1.0):  {len(active)}  ({100*len(active)/len(df):.1f}%)')
print(f'Weak MJO:                  {df["weak_mjo"].sum()}')
print()

print('ENSO distribution (all days):')
for cat, n in df['enso_category'].value_counts().items():
    print(f'  {cat:10s}: {n:5d} days ({100*n/len(df):.1f}%)')
print()

print('ENSO distribution (active MJO days only):')
for cat, n in active['enso_category'].value_counts().items():
    print(f'  {cat:10s}: {n:5d} days ({100*n/len(active):.1f}%)')
print()

print('Phase distribution (active MJO, phases 1-8):')
pc = active['phase'].value_counts().sort_index()
for ph, n in pc.items():
    if ph <= 8:
        print(f'  Phase {ph}: {n:4d} days')
print()

print('Phase × ENSO cross-table (active MJO days):')
ct = pd.crosstab(active[active['phase'] <= 8]['phase'],
                 active[active['phase'] <= 8]['enso_category'])
print(ct)
print('\n(Check: no zero cells in this table)')
print('=' * 60)

## Cell 9 — Verification: Phase Composite Polar Plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

df_plot = pd.read_csv(out_path, parse_dates=['date'])
active  = df_plot[~df_plot['weak_mjo']]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Left: scatter colored by phase ---
ax = axes[0]
cmap = cm.get_cmap('tab10', 8)
for ph in range(1, 9):
    sub = active[active['phase'] == ph]
    ax.scatter(sub['rmm1'], sub['rmm2'], s=2, alpha=0.3,
               color=cmap(ph - 1), label=f'Phase {ph}')
# Draw phase octant boundaries
for i in range(8):
    angle = np.radians(i * 45)
    ax.plot([0, 3 * np.cos(angle)], [0, 3 * np.sin(angle)],
            'k--', lw=0.5, alpha=0.4)
# Unit circle
theta = np.linspace(0, 2 * np.pi, 300)
ax.plot(np.cos(theta), np.sin(theta), 'k-', lw=1, alpha=0.4)
ax.set_xlabel('RMM1', fontsize=12)
ax.set_ylabel('RMM2', fontsize=12)
ax.set_title('RMM phase space (active MJO, 1979-2023)', fontsize=12, fontweight='bold')
ax.set_aspect('equal')
ax.legend(fontsize=8, markerscale=4, loc='lower right')
ax.axhline(0, color='k', lw=0.5, alpha=0.3)
ax.axvline(0, color='k', lw=0.5, alpha=0.3)

# --- Right: phase distribution bar, stratified by ENSO ---
ax2 = axes[1]
enso_colors = {'El Nino': '#d62728', 'Neutral': '#aec7e8', 'La Nina': '#1f77b4'}
phases = list(range(1, 9))
bottoms = np.zeros(8)
for cat in ['El Nino', 'Neutral', 'La Nina']:
    sub = active[active['enso_category'] == cat]
    counts = [len(sub[sub['phase'] == ph]) for ph in phases]
    ax2.bar(phases, counts, bottom=bottoms, label=cat,
            color=enso_colors[cat], alpha=0.85)
    bottoms += np.array(counts)
ax2.set_xticks(phases)
ax2.set_xlabel('RMM Phase', fontsize=12)
ax2.set_ylabel('Days', fontsize=12)
ax2.set_title('Phase distribution by ENSO (active MJO)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)

plt.tight_layout()
fig_path = f'{MJO_DIR}/results/rmm_phase_enso_distribution.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## Cell 10 — Seasonal ENSO Balance Check

Checks that ENSO is reasonably balanced across all four seasons (important for all-year scope).

In [ ]:
df = pd.read_csv(out_path, parse_dates=['date'])

season_map = {12: 'DJF', 1: 'DJF', 2: 'DJF',
              3:  'MAM', 4: 'MAM', 5: 'MAM',
              6:  'JJA', 7: 'JJA', 8: 'JJA',
              9:  'SON', 10:'SON', 11:'SON'}
df['season'] = df['date'].dt.month.map(season_map)

print('ENSO balance by season (all days, 1979-2023):')
ct = pd.crosstab(df['season'], df['enso_category'])
ct = ct.loc[['DJF', 'MAM', 'JJA', 'SON']]  # canonical order
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
print(ct)
print()
print('As percentages:')
print(ct_pct.round(1))

print()
print('Amplitude statistics by ENSO category:')
print(df.groupby('enso_category')['amplitude'].describe().round(3))

print()
print('Done. rmm_labels.csv is ready for nb13 (preprocessing).')

---
## Done!

Google Drive should now contain:

```
BSISO_SSL_Project/MJO/
├── data/
│   ├── raw/
│   │   ├── rmm_raw.txt            ← raw BoM download
│   │   ├── nino34_monthly.txt     ← NOAA Nino3.4 monthly
│   │   └── rmm_labels.csv         ← output of this notebook ✓
│   └── processed/
├── checkpoints/
└── results/
    ├── sup/
    ├── ssl/
    └── rmm_phase_enso_distribution.png
```

**Next steps (can run in parallel):**
- **nb12** (`12_mjo_era5_download.ipynb`) — download u850, u200, OLR from ERA5 CDS
- nb12 and nb11 (this notebook) feed into **nb13** (preprocessing)

---
*DDCS Project | jh9141@nyu.edu*